# Movie Recommendation Systems in Python

This notebook implements and compares three recommendation strategies on movie data.

This notebook walks through:

1. **A popularity-based baseline** using weighted ratings and popularity
2. **Memory-based collaborative filtering** using movie-to-movie correlation
3. **K-nearest neighbours (KNN)** using cosine similarity on a sparse user-item matrix

The goal is not just to generate recommendations, but to show how different recommendation paradigms behave, what assumptions they make, and when each one is useful.

---

## Why this project matters

Recommendation systems are a core data science application because they combine:

- data cleaning and transformation
- feature engineering
- matrix construction
- similarity measurement
- model interpretation

This notebook is intentionally structured to move from a **simple, interpretable baseline** to more **personalized collaborative approaches**.

## 1. Popularity-based recommendation baseline

A popularity model is a useful starting point because it does not require user-level personalization.  
It answers a practical question:

> *Which movies should I recommend if I only know what tends to perform well overall?*

In this section, I use the TMDB metadata dataset to create a baseline recommender that combines:

- **vote quality** (`vote_average`)
- **rating volume** (`vote_count`)
- **platform interest** (`popularity`)

This gives a strong benchmark before moving to collaborative filtering.

In [ ]:
import kagglehub

# Download the TMDB metadata used for the popularity-based baseline.
path = kagglehub.dataset_download("tmdb/tmdb-movie-metadata")

print("Path to dataset files:", path)


### Load the dataset

The first step is to download the TMDB metadata and read the two tables used in this section:

- **movies**: title-level movie attributes
- **credits**: cast and crew metadata

Even though the credits table is not yet used for content-based modelling here, it is merged in so the combined movie table is easier to extend later.

In [ ]:
# Load the TMDB movie metadata and credits tables.
import os
import pandas as pd

movies_df = pd.read_csv(os.path.join(path, "tmdb_5000_movies.csv"))
credits = pd.read_csv(os.path.join(path, "tmdb_5000_credits.csv"))


In [ ]:
print("Credits:",credits.shape)
print("Movies Dataframe:",movies_df.shape)

### Prepare a single analysis table

The credits table uses `movie_id`, while the movies table uses `id`, so the identifier is first aligned before merging.

In [ ]:
credits_column_renamed = credits.rename(index=str, columns={"movie_id": "id"})

In [ ]:
credits_column_renamed.head()

In [ ]:
movies_df_merge = movies_df.merge(credits_column_renamed, on='id')

In [ ]:
movies_df_merge.head()

### Clean the merged movie table

For the popularity-based baseline, several columns are not needed.  
Removing them keeps the working table more focused and easier to inspect.

In [ ]:
# Remove columns that are not needed for the popularity-based scoring workflow.
movies_cleaned_df = movies_df_merge.drop(columns=['homepage', 'title_x', 'title_y', 'status','production_countries'])
movies_cleaned_df.head()


In [ ]:
movies_cleaned_df.info()

### Compute the weighted rating components

A plain average rating can be misleading.  
For example, a movie with a single 10/10 rating should not outrank a movie with thousands of strong ratings.

To make the ranking more robust, this notebook uses the standard weighted rating formulation:

\[
\text{Weighted Rating} = \frac{R \cdot v + C \cdot m}{v + m}
\]

where:

- \(R\) = average rating for the movie
- \(v\) = number of votes for the movie
- \(C\) = mean rating across the whole dataset
- \(m\) = minimum votes required to be considered reliable

This shrinks movies with low vote counts toward the global average, making the ranking more stable.

In [ ]:
# Components used in the weighted rating formula.
v=movies_cleaned_df['vote_count']
R=movies_cleaned_df['vote_average']
C=movies_cleaned_df['vote_average'].mean()
m=movies_cleaned_df['vote_count'].quantile(0.70)


In [ ]:
movies_cleaned_df['weighted_average']=((R*v)+ (C*m))/(v+m)

In [ ]:
movies_cleaned_df.head()

### Rank movies by weighted score

After calculating the weighted score, the movies can be ordered from strongest to weakest according to this baseline metric.

In [ ]:
movie_sorted_ranking=movies_cleaned_df.sort_values('weighted_average',ascending=False)
movie_sorted_ranking[['original_title', 'vote_count', 'vote_average', 'weighted_average', 'popularity']].head(20)

### Visual inspection of top weighted titles

A quick chart makes it easier to validate whether the weighted ranking is surfacing well-known, highly-rated films.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

weight_average=movie_sorted_ranking.sort_values('weighted_average',ascending=False)
plt.figure(figsize=(12,6))
axis1=sns.barplot(x=weight_average['weighted_average'].head(10), y=weight_average['original_title'].head(10), data=weight_average)
plt.xlim(4, 10)
plt.title('Best Movies by average votes', weight='bold')
plt.xlabel('Weighted Average Score', weight='bold')
plt.ylabel('Movie Title', weight='bold')

In [ ]:
# Visualize the most popular movies.
popularity=movie_sorted_ranking.sort_values('popularity',ascending=False)
plt.figure(figsize=(12,6))
ax=sns.barplot(x=popularity['popularity'].head(10), y=popularity['original_title'].head(10), data=popularity)

plt.title('Most Popular by Votes', weight='bold')
plt.xlabel('Score of Popularity', weight='bold')
plt.ylabel('Movie Title', weight='bold')


In [ ]:
popularity.head()

### Blend rating quality with popularity

A recommendation list can benefit from balancing:

- **quality**: strong weighted ratings
- **reach / interest**: high popularity

Because these features are on different scales, they are first normalized with `MinMaxScaler`, then combined with equal weight.

In [ ]:
# Blend normalized weighted rating and popularity with equal importance.
from sklearn.preprocessing import MinMaxScaler

scaling=MinMaxScaler()
movie_scaled_df=scaling.fit_transform(movies_cleaned_df[['weighted_average','popularity']])
movie_normalized_df=pd.DataFrame(movie_scaled_df,columns=['weighted_average','popularity'])
movie_normalized_df.head()


In [ ]:
movies_cleaned_df[['normalized_weight_average','normalized_popularity']]= movie_normalized_df

In [ ]:
movies_cleaned_df.head()

The final blended score below is a simple but practical business-style ranking metric:

- 50% normalized weighted rating
- 50% normalized popularity

In [ ]:
movies_cleaned_df['score'] = movies_cleaned_df['normalized_weight_average'] * 0.5 + movies_cleaned_df['normalized_popularity'] * 0.5
movies_scored_df = movies_cleaned_df.sort_values(['score'], ascending=False)
movies_scored_df[['original_title', 'normalized_weight_average', 'normalized_popularity', 'score']].head(20)

In [ ]:
scored_df = movies_cleaned_df.sort_values('score', ascending=False)

plt.figure(figsize=(16,6))

ax = sns.barplot(x=scored_df['score'].head(10), y=scored_df['original_title'].head(10), data=scored_df, palette='deep')

#plt.xlim(3.55, 5.25)
plt.title('Best Rated & Most Popular Blend', weight='bold')
plt.xlabel('Score', weight='bold')
plt.ylabel('Movie Title', weight='bold')

---

## 2. Collaborative filtering with movie-to-movie correlation

Popularity-based recommenders are useful, but they are not personalized.  
Collaborative filtering improves on this by learning from **patterns in user behaviour**.

In this section, the recommendation logic becomes:

> *Users who rated one movie in a certain way often rated similar movies in related ways.*

The [MovieLens 100K dataset](https://grouplens.org/datasets/movielens/100k/) is used here because it contains explicit user ratings, which makes it ideal for demonstrating user-item interactions. Review the GroupLens usage terms and citation requirements before reuse.

### Load MovieLens ratings data

The official archive is downloaded into an ignored local cache and verified against the published checksum. After extracting it, I load:

- `u.data`: user-movie ratings
- `u.item`: movie ID to movie title mapping

In [ ]:
# Download and verify the official MovieLens 100K archive.
import hashlib
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

archive_url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
archive_path = Path("datasets/.cache/ml-100k.zip")
extract_dir = Path("datasets/movie-lens-dataset-100k")
expected_md5 = "0e33842e24a9c977be4e0107933c0723"

archive_path.parent.mkdir(parents=True, exist_ok=True)
if not archive_path.exists():
    urlretrieve(archive_url, archive_path)

actual_md5 = hashlib.md5(archive_path.read_bytes()).hexdigest()
if actual_md5 != expected_md5:
    raise ValueError(f"MovieLens archive checksum mismatch: {actual_md5}")

if not (extract_dir / "ml-100k").exists():
    with zipfile.ZipFile(archive_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)


In [ ]:
column_names = ['user_id', 'item_id', 'rating', 'timestamp']
file_path = os.path.join("datasets", "movie-lens-dataset-100k", "ml-100k", "u.data")
df = pd.read_csv(file_path, sep='\t', names=column_names)

In [ ]:
df.head()

In [ ]:
file_path = os.path.join("datasets", "movie-lens-dataset-100k", "ml-100k", "u.item")
movie_titles = pd.read_csv(file_path, sep='|', encoding='latin-1', header=None, usecols=[0, 1], names=['item_id', 'title'])
movie_titles.head()

In [ ]:
df = pd.merge(df,movie_titles,on='item_id')
df.head()

### Build the user-item matrix

The pivot table below restructures the transactional ratings data into a matrix where:

- each **row** represents a user
- each **column** represents a movie
- each **value** is the rating that user gave that movie

This matrix is the foundation for collaborative filtering.

In [ ]:
# Build a user-item ratings matrix for movie-to-movie collaborative filtering.
moviemat = df.pivot_table(index='user_id', columns='title', values='rating')
moviemat.head()


### Find similar movies using Pearson correlation

Here I use **Star Wars (1977)** as an example query movie.

The idea is to compare its ratings vector against every other movie column.  
If two movies tend to be rated similarly by the same users, they will have a higher correlation.

In [ ]:
# Use movie-to-movie correlation to find similar movies based on user ratings.
# Here, Star Wars is used as a simple example query movie.

starwars_user_ratings = moviemat['Star Wars (1977)']

similar_to_starwars = moviemat.corrwith(starwars_user_ratings)


In [ ]:
similar_to_starwars

The resulting correlation series is converted into a DataFrame so it can be cleaned, inspected, and sorted more easily.

In [ ]:
corr_starwars = pd.DataFrame(similar_to_starwars,columns=['Correlation'])
corr_starwars.dropna(inplace=True)
corr_starwars.head()

In [ ]:
corr_starwars.sort_values('Correlation',ascending=False).head(10)

### Add rating volume for reliability

Correlation alone is not enough.  
A movie might appear highly correlated simply because only a small number of users rated both titles.

To reduce this noise, I calculate the number of ratings per movie and join that information back to the correlation results.

In [ ]:
ratings = pd.DataFrame(df.groupby('title')['rating'].mean())
ratings['num of ratings'] = pd.DataFrame(df.groupby('title')['rating'].count())

In [ ]:
ratings.head()

In [ ]:
# Filter recommendations to titles with a meaningful amount of rating activity.
corr_starwars = corr_starwars.join(ratings['num of ratings'])
corr_starwars.head()


Filtering to movies with more than 100 ratings gives a more credible set of neighbours for the query title.

In [ ]:
corr_starwars[corr_starwars['num of ratings']>100].sort_values('Correlation',ascending=False).head()

---

## 3. K-nearest neighbours (KNN) with cosine similarity

Correlation-based collaborative filtering is a classic and interpretable approach.  
Another common alternative is to represent each movie as a vector of user ratings and then retrieve the closest movies with a nearest-neighbour model.

In this section:

- movies are represented as rows in a user-rating feature space
- missing ratings are filled with zeroes
- the matrix is converted to a sparse representation
- `NearestNeighbors` is used with **cosine similarity** (via cosine distance)

This is a practical way to scale similarity search over large, sparse interaction matrices.

### Reload the MovieLens data for the KNN workflow

This section uses the same underlying ratings dataset, but the preprocessing is arranged for nearest-neighbour search rather than direct correlation.

In [ ]:
file_path = os.path.join("datasets", "movie-lens-dataset-100k", "ml-100k", "u.item")
movies_df = pd.read_csv(file_path, sep='|', encoding='latin-1', header=None, usecols=[0, 1], names=['movieId', 'title'])
movies_df.head()

In [ ]:
file_path = os.path.join("datasets", "movie-lens-dataset-100k", "ml-100k", "u.data")
rating_df = pd.read_csv(file_path, sep='\t', usecols=[0, 1, 2], names=['userId', 'movieId', 'rating'])
rating_df.head()

In [ ]:
df = pd.merge(rating_df, movies_df, on='movieId')
df.head()

### Prepare the ratings table

The next few cells:

1. remove missing titles
2. count how many ratings each movie received
3. filter out low-support movies
4. build a movie-user matrix

This helps the KNN model focus on movies with enough interaction history to produce meaningful neighbours.

In [ ]:
# Remove rows where the title is missing before building the KNN input matrix.
combine_movie_rating = df.dropna(axis = 0, subset = ['title'])


In [ ]:
combine_movie_rating

In [ ]:
movie_ratingCount = (combine_movie_rating.
     groupby(by = ['title'])['rating'].
     count().
     reset_index().
     rename(columns = {'rating': 'totalRatingCount'})
     [['title', 'totalRatingCount']]
    )
movie_ratingCount.head()


In [ ]:
rating_with_totalRatingCount = combine_movie_rating.merge(movie_ratingCount, left_on = 'title', right_on = 'title', how = 'left')
rating_with_totalRatingCount.head()

In [ ]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)
print(movie_ratingCount['totalRatingCount'].describe())

In [ ]:
# Keep only movies with enough ratings to reduce noise in nearest-neighbour search.
popularity_threshold = 50
rating_popular_movie= rating_with_totalRatingCount.query('totalRatingCount >= @popularity_threshold')
rating_popular_movie.head()


### Create the movie-user feature matrix

Compared with the earlier collaborative filtering section, the orientation changes here:

- each **row** is now a movie
- each **column** is a user
- the values are ratings, with missing values filled as 0

This turns each movie into a feature vector that can be compared to other movies.

In [ ]:
movie_features_df=rating_popular_movie.pivot_table(index='title',columns='userId',values='rating').fillna(0)
movie_features_df.head()

### Convert to sparse format and fit KNN

Ratings matrices are mostly empty because most users rate only a small subset of movies.  
A sparse matrix stores this efficiently.

`NearestNeighbors(metric='cosine', algorithm='brute')` is then trained so the notebook can retrieve similar movie vectors based on cosine distance.

In [ ]:
from scipy.sparse import csr_matrix

movie_features_df_matrix = csr_matrix(movie_features_df.values)

from sklearn.neighbors import NearestNeighbors

model_knn = NearestNeighbors(metric = 'cosine', algorithm = 'brute')
model_knn.fit(movie_features_df_matrix)


In [ ]:
movie_features_df_matrix

### Generate example recommendations

To demonstrate the fitted KNN model, a reproducibly sampled movie is selected and its nearest neighbours are retrieved.

Lower distance means the neighbour is more similar in rating behaviour to the query movie.

In [ ]:
import numpy as np

# Pick a reproducible movie vector and retrieve its nearest neighbours.
rng = np.random.default_rng(42)
query_index = rng.integers(movie_features_df.shape[0])
print(query_index)
distances, indices = model_knn.kneighbors(movie_features_df.iloc[query_index,:].values.reshape(1, -1), n_neighbors = 6)


In [ ]:
for i in range(0, len(distances.flatten())):
    if i == 0:
        print('Recommendations for {0}:\n'.format(movie_features_df.index[query_index]))
    else:
        print('{0}: {1}, with distance of {2}:'.format(i, movie_features_df.index[indices.flatten()[i]], distances.flatten()[i]))


---

## Final takeaway

This notebook demonstrates three increasingly sophisticated recommendation strategies:

- **Popularity-based ranking** for a strong, interpretable baseline
- **Correlation-based collaborative filtering** for memory-based similarity
- **KNN on sparse user-item data** for nearest-neighbour recommendations

Together, they show an end-to-end recommendation workflow that covers:

- data preparation
- ranking logic
- user-item matrix construction
- similarity-based retrieval
- practical filtering to improve recommendation quality

For a production system, the next natural extensions would be:

- content-based features using genres, cast, or keywords
- matrix factorization / latent factor models
- offline evaluation with ranking metrics
- hybrid recommendation strategies
